Soal 1 – BFS dan Dijkstra (SSSP)

In [1]:
from collections import deque
import heapq

# ============================
# Adjacency List
# ============================
graph = {
    'A': [('B', 4), ('C', 2)],
    'B': [('A', 4), ('C', 1), ('D', 5)],
    'C': [('A', 2), ('B', 1), ('D', 8), ('E', 10)],
    'D': [('B', 5), ('C', 8), ('E', 2), ('F', 6)],
    'E': [('C', 10), ('D', 2), ('F', 3)],
    'F': [('D', 6), ('E', 3)]
}

# ============================
# Breadth First Search (BFS)
# ============================
def bfs(graph, start):
    visited = set()
    queue = deque([start])
    order = []

    visited.add(start)

    while queue:
        current = queue.popleft()
        order.append(current)

        for neighbor, weight in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order


# ============================
# Dijkstra
# ============================
def dijkstra(graph, start):
    distance = {node: float('inf') for node in graph}
    previous = {node: None for node in graph}

    distance[start] = 0

    pq = [(0, start)]

    while pq:
        current_distance, current_node = heapq.heappop(pq)

        if current_distance > distance[current_node]:
            continue

        for neighbor, weight in graph[current_node]:
            new_distance = current_distance + weight

            if new_distance < distance[neighbor]:
                distance[neighbor] = new_distance
                previous[neighbor] = current_node
                heapq.heappush(pq, (new_distance, neighbor))

    return distance, previous


# ============================
# Menampilkan Path
# ============================
def shortest_path(previous, start, target):
    path = []

    while target is not None:
        path.append(target)
        target = previous[target]

    path.reverse()

    return path


# ============================
# Program Utama
# ============================

print("===== BFS Traversal =====")
order = bfs(graph, 'A')
print(" -> ".join(order))

distance, previous = dijkstra(graph, 'A')

print("\n===== Shortest Distance =====")
for node in sorted(distance):
    print(f"A -> {node} = {distance[node]}")

print("\n===== Shortest Path A ke F =====")
path = shortest_path(previous, 'A', 'F')
print(" -> ".join(path))
print("Total Distance =", distance['F'])

===== BFS Traversal =====
A -> B -> C -> D -> E -> F

===== Shortest Distance =====
A -> A = 0
A -> B = 3
A -> C = 2
A -> D = 8
A -> E = 10
A -> F = 13

===== Shortest Path A ke F =====
A -> C -> B -> D -> E -> F
Total Distance = 13


Soal 2 – Bellman-Ford dan Floyd-Warshall

In [2]:
# ==========================================
# Edge List
# ==========================================

vertices = ['A', 'B', 'C', 'D', 'E']

edges = [
    ('A', 'B', 4),
    ('A', 'C', 2),
    ('B', 'C', -2),
    ('B', 'D', 3),
    ('C', 'D', 2),
    ('D', 'E', 2),
    ('E', 'B', -1)
]

# ==========================================
# Bellman-Ford Algorithm
# ==========================================

def bellman_ford(vertices, edges, source):
    distance = {v: float('inf') for v in vertices}
    previous = {v: None for v in vertices}

    distance[source] = 0

    # Relaksasi sebanyak |V|-1 kali
    for i in range(len(vertices) - 1):
        for u, v, w in edges:
            if distance[u] != float('inf') and distance[u] + w < distance[v]:
                distance[v] = distance[u] + w
                previous[v] = u

    # Deteksi Negative Cycle
    for u, v, w in edges:
        if distance[u] != float('inf') and distance[u] + w < distance[v]:
            print("Graf memiliki Negative Cycle")
            return None, None

    return distance, previous


# ==========================================
# Menampilkan Path
# ==========================================

def get_path(previous, target):
    path = []

    while target is not None:
        path.append(target)
        target = previous[target]

    path.reverse()
    return path


# ==========================================
# Floyd-Warshall Algorithm
# ==========================================

def floyd_warshall(vertices, edges):

    n = len(vertices)

    index = {vertices[i]: i for i in range(n)}

    INF = float('inf')

    dist = [[INF] * n for _ in range(n)]

    for i in range(n):
        dist[i][i] = 0

    for u, v, w in edges:
        i = index[u]
        j = index[v]
        dist[i][j] = w

    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]

    return dist, index


# ==========================================
# Main Program
# ==========================================

distance, previous = bellman_ford(vertices, edges, 'A')

print("===== Bellman-Ford =====")

for v in vertices:
    print(f"A -> {v} = {distance[v]}")

print("\nShortest Path A ke E")

path = get_path(previous, 'E')

print(" -> ".join(path))

print("\n")

dist, index = floyd_warshall(vertices, edges)

print("===== Floyd Warshall =====")

print("     ", end="")

for v in vertices:
    print(f"{v:>5}", end="")

print()

for i in range(len(vertices)):
    print(f"{vertices[i]:>5}", end="")

    for j in range(len(vertices)):

        if dist[i][j] == float('inf'):
            print(f"{'INF':>5}", end="")
        else:
            print(f"{dist[i][j]:>5}", end="")

    print()

===== Bellman-Ford =====
A -> A = 0
A -> B = 4
A -> C = 2
A -> D = 4
A -> E = 6

Shortest Path A ke E
A -> C -> D -> E


===== Floyd Warshall =====
         A    B    C    D    E
    A    0    4    2    4    6
    B  INF    0   -2    0    2
    C  INF    3    0    2    4
    D  INF    1   -1    0    2
    E  INF   -1   -3   -1    0


Soal 3 – Minimum Spanning Tree (Prim/Kruskal)

In [3]:
import heapq

# ==========================================
# Data Graf
# ==========================================

vertices = ['A','B','C','D','E','F','G','H','I']

edges = [
    ('A','B',4),
    ('A','H',8),
    ('B','H',11),
    ('B','C',8),
    ('C','D',7),
    ('C','I',2),
    ('C','F',4),
    ('D','E',9),
    ('D','F',14),
    ('E','F',10),
    ('F','G',2),
    ('G','H',1),
    ('G','I',6),
    ('H','I',7)
]

# ==========================================
# Membuat Adjacency List
# ==========================================

graph = {}

for v in vertices:
    graph[v] = []

for u, v, w in edges:
    graph[u].append((v, w))
    graph[v].append((u, w))


# ==========================================
# Algoritma Prim
# ==========================================

def prim(graph, start):

    visited = set()

    pq = []

    mst = []

    total_cost = 0

    visited.add(start)

    for neighbor, weight in graph[start]:
        heapq.heappush(pq, (weight, start, neighbor))

    while pq:

        weight, u, v = heapq.heappop(pq)

        if v in visited:
            continue

        visited.add(v)

        mst.append((u, v, weight))

        total_cost += weight

        for next_vertex, next_weight in graph[v]:

            if next_vertex not in visited:
                heapq.heappush(
                    pq,
                    (next_weight, v, next_vertex)
                )

    return mst, total_cost


# ==========================================
# Disjoint Set (Union Find)
# ==========================================

parent = {}

rank = {}

def make_set(vertices):

    for v in vertices:
        parent[v] = v
        rank[v] = 0

def find(v):

    if parent[v] != v:
        parent[v] = find(parent[v])

    return parent[v]

def union(v1, v2):

    root1 = find(v1)
    root2 = find(v2)

    if root1 == root2:
        return

    if rank[root1] < rank[root2]:
        parent[root1] = root2

    elif rank[root1] > rank[root2]:
        parent[root2] = root1

    else:
        parent[root2] = root1
        rank[root1] += 1


# ==========================================
# Algoritma Kruskal
# ==========================================

def kruskal(vertices, edges):

    make_set(vertices)

    mst = []

    total_cost = 0

    edges = sorted(edges, key=lambda x: x[2])

    for u, v, weight in edges:

        if find(u) != find(v):

            union(u, v)

            mst.append((u, v, weight))

            total_cost += weight

    return mst, total_cost


# ==========================================
# Program Utama
# ==========================================

print("===== PRIM =====")

mst, cost = prim(graph, 'A')

for u, v, w in mst:
    print(f"{u} - {v} = {w}")

print("Total Cost =", cost)


print("\n===== KRUSKAL =====")

mst, cost = kruskal(vertices, edges)

for u, v, w in mst:
    print(f"{u} - {v} = {w}")

print("Total Cost =", cost)

===== PRIM =====
A - B = 4
A - H = 8
H - G = 1
G - F = 2
F - C = 4
C - I = 2
C - D = 7
D - E = 9
Total Cost = 37

===== KRUSKAL =====
G - H = 1
C - I = 2
F - G = 2
A - B = 4
C - F = 4
C - D = 7
A - H = 8
D - E = 9
Total Cost = 37
